In [7]:
pip install transformers datasets torch accelerate streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 133.9 MB/s eta 0:00:00


In [8]:
import transformers
import datasets
print("Setup OK")

Setup OK


In [9]:
from datasets import load_dataset

dataset = load_dataset("bdotloh/empathetic-dialogues-contexts")

print(dataset)
print("\nExample entry:")
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'situation', 'emotion'],
        num_rows: 19209
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'situation', 'emotion'],
        num_rows: 2756
    })
    test: Dataset({
        features: ['Unnamed: 0', 'situation', 'emotion'],
        num_rows: 2542
    })
})

Example entry:
{'Unnamed: 0': 0, 'situation': 'I remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.', 'emotion': 'sentimental'}


In [10]:
# Check column names
print(dataset['train'].column_names)
print("\nExample entry:")
print(dataset['train'][0])

['Unnamed: 0', 'situation', 'emotion']

Example entry:
{'Unnamed: 0': 0, 'situation': 'I remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.', 'emotion': 'sentimental'}


In [11]:
from datasets import load_dataset

# Load a better dataset with actual conversations
dataset = load_dataset("Amod/mental_health_counseling_conversations")

print(dataset)
print("\nExample entry:")
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response'],
        num_rows: 3512
    })
})

Example entry:
{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life

In [12]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

# Preprocess function
def preprocess(example):
    input_text = "User: " + example["Context"] + " Bot: " + example["Response"]
    tokenized = tokenizer(
        input_text,
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Apply preprocessing
tokenized_dataset = dataset.map(preprocess, remove_columns=["Context", "Response"])

print("Preprocessing done!")
print(tokenized_dataset)

Preprocessing done!
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3512
    })
})


In [13]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Load model
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Training arguments
training_args = TrainingArguments(
    output_dir="./mental_health_chatbot",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    logging_steps=100,
    learning_rate=5e-5,
    fp16=True,
    report_to="none"
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

# Start training
trainer.train()

print("Training done!")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.363192
200,3.197702
300,3.191842
400,3.137836
500,3.100638
600,3.044111
700,2.954037
800,2.962215
900,2.972160
1000,2.861545


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training done!


In [14]:
# Save the model and tokenizer
model.save_pretrained("./mental_health_chatbot_model")
tokenizer.save_pretrained("./mental_health_chatbot_model")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [15]:
import os

files = os.listdir("./mental_health_chatbot_model")
print("Saved files:", files)

Saved files: ['tokenizer.json', 'generation_config.json', 'tokenizer_config.json', 'config.json', 'model.safetensors']


In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load saved model
model = AutoModelForCausalLM.from_pretrained("./mental_health_chatbot_model")
tokenizer = AutoTokenizer.from_pretrained("./mental_health_chatbot_model")

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Chat function
def chat(user_input):
    input_text = "User: " + user_input + " Bot:"
    inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs,
        max_length=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    bot_response = response.split("Bot:")[-1].strip()
    return bot_response

# Run chatbot loop
print("Mental Health Support Chatbot")
print("Type 'quit' to exit\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        print("Take care of yourself. Goodbye!")
        break
    response = chat(user_input)
    print(f"Bot: {response}\n")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Mental Health Support Chatbot
Type 'quit' to exit

You: I am feeling stressed today
Bot: I am sorry to hear of the stressor you are experiencing today. It is often the result of a lack of understanding of what is really going on, and I am sorry you are feeling stressed today. I am not sure if you are experiencing stress disorder, depression, or something similar. If so, I would encourage you to seek professional help to help you manage the stress. In the meantime, I encourage you to seek professional help to help you manage the stress. There are several ways to manage stress:1. You may want to talk with your therapist about your concerns and concerns.2. You may want to talk to your therapist about the stress.3. You may want to talk with your therapist about the stress.4. You may want to talk to your therapist about the stress.5. You may want to talk to your therapist about the stress.6. You may want to talk to your therapist about the stress.7. You

You: quit
Take care of yourself. Goo

In [17]:
!pip install streamlit pyngrok -q

In [18]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model
@st.cache_resource
def load_model():
    model = AutoModelForCausalLM.from_pretrained("./mental_health_chatbot_model")
    tokenizer = AutoTokenizer.from_pretrained("./mental_health_chatbot_model")
    return model, tokenizer

model, tokenizer = load_model()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Chat function
def chat(user_input):
    input_text = "User: " + user_input + " Bot:"
    inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
    outputs = model.generate(
        inputs,
        max_length=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Bot:")[-1].strip()

# Streamlit UI
st.title("🧠 Mental Health Support Chatbot")
st.write("I'm here to listen and support you. Feel free to share how you're feeling.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("How are you feeling today?")
if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)
    response = chat(user_input)
    st.session_state.messages.append({"role": "assistant", "content": response})
    with st.chat_message("assistant"):
        st.write(response)

Writing app.py


In [20]:
from google.colab import output
import subprocess

# Start Streamlit
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])

# Open in Colab
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>